In [ ]:
import os
from dotenv import load_dotenv
import redcap
import pandas as pd
import pandasql as psql
from datetime import datetime, timedelta
import re
from sklearn.metrics.pairwise import cosine_similarity

# Load environment variables
load_dotenv()

# Initialize REDCap projects
df = redcap.Project(
    os.getenv('REDCAP_MAIN_URL'),
    os.getenv('REDCAP_MAIN_TOKEN')
)

ck_wk = redcap.Project(
    os.getenv('REDCAP_CK_WK_URL'),
    os.getenv('REDCAP_CK_WK_TOKEN')
)

indigo = redcap.Project(
    os.getenv('REDCAP_INDIGO_URL'),
    os.getenv('REDCAP_INDIGO_TOKEN')
)

In [2]:
sen_data=ck_wk.export_report(report_id='10068')
sen_log_data=pd.DataFrame(sen_data)
sen_log_data=sen_log_data[['wk_ckno','enu_name','enu_village', 'enu_comp','sen_contact1','sen_contact2','sen_contact3']]
sen_log_data=sen_log_data.rename(columns={'wk_ckno': 'ck_wkno'})
village_mapping = {
'001':'Dumbuto',
'002':'Sankandi',
'003':'Nioro Jattaba',
'004':'Jattaba',
'005':'Jiffarong',
'006':'Bajana',
'007':'Kuli Kunda',
'008':'Jamaru',
'009':'Brikamanding',
'010':'Kantong Kunda',
'011':'Jali',
'013':'Manduar',
'014':'Bang Kuling',
'015':'Gissay',
'016':'Tankular',
'017':'Joli',
'018':'Kuyang',
'019':'Bantasu',
'020':'Santamba',
'021':'Missira',
'022':'Taborangkoto',
'023':'Burong',
'024':'Jula Kunda',
'025':'Karantaba',
'026':'Mandina',
'027':'Janneh Kunda',
'028':'Kemoto',
'029':'Keneba',
'030':'Batelling',
'031':'Sandeng',
'032':'Wudeba',
'034':'Kenokoto',
'035':'Manari',
'036':'Nineteen',
'040':'WUROKANG',
'041':'KWINELLA SANSANKONO',
'042':'KWINELLA NIA KUNDA',
'043':'TENDABA',
'044':'BUMARR',
'045':'BAMBAKO',
'046':'KUNDONG MARIAYA',
'047':'NEMA',
'048':'KUNDANG NUMU KUNDA',
'049':'KUNDANG FULA KUNDA',
'050':'NEMA KUTA',
'051':'JIRROFF',
'052':'MADINA ANGALLEH',
'053':'JATTA KUNDA',
'054':'MANDINA CENTRAL',
'055':'SARE SARJO',
'056':'SIBETO',
'057':'SARE NDALLA',
'058':'TABANANI',
'060':'WILLINGARA',
'061':'SARE MAMUDU'
    # Add more mappings as needed
    # Add more mappings as needed
}

# Assuming sen_log_data is your DataFrame
sen_log_data['enu_village'] = sen_log_data['enu_village'].map(village_mapping)

In [3]:
df_ultrasound_scan=df.export_records(forms=['ultrasound_scan'])

In [4]:
df_ultrasound_scan=pd.DataFrame(df_ultrasound_scan)

In [5]:
df_ultrasound_scan=df_ultrasound_scan[['participant_id','ultrascan_date','ultrascan_exp_date']]
df_ultrasound_scan=df_ultrasound_scan[
    (df_ultrasound_scan['ultrascan_exp_date']!='')
]

In [ ]:
df_ultrasound_scan=df_ultrasound_scan[
    (df_ultrasound_scan['ultrascan_exp_date']>='2024-05-01')&
    (df_ultrasound_scan['ultrascan_exp_date']<='2025-04-31')
].drop_duplicates('participant_id')

In [7]:
ck_v4_Ab=df.export_report(report_id='10375')
ck_v4_Ab=pd.DataFrame(ck_v4_Ab)
ck_v4_Ab_merg=pd.merge(ck_v4_Ab,df_ultrasound_scan,on='participant_id',how='inner').drop_duplicates(subset='participant_id')
df_ultrasound_scan_eddlist=ck_v4_Ab_merg[['participant_id','ck_wkno','ultrascan_exp_date']]

In [8]:
eddlist=pd.merge(sen_log_data,df_ultrasound_scan_eddlist,on='ck_wkno',how='inner')
eddlist.to_csv('eddlist.csv',index=False)